## Stage 2 Unlearning - All 4 Methods

Paper: FIUBench (ICLR 2025)  
Goal: Train GA, GD, KL, PO unlearning on Stage 1 checkpoint using exact hyperparameters from Table 7.

| Method | forget_loss | lr   | batch | accum | epochs | split   |
|--------|-------------|------|-------|-------|--------|---------|
| GA     | ga          | 2e-5 | 8     | 16    | 8      | forget5 |
| GD     | gd          | 2e-5 | 8     | 16    | 8      | forget5 |
| KL     | kl          | 1e-4 | 8     | 16    | 8      | forget5 |
| PO     | idk         | 3e-4 | 8     | 16    | 8      | forget5 |

Run cells sequentially. Each method saves its checkpoint to Drive before the next begins.


## GitHub Repo

In [1]:
import subprocess

result = subprocess.run(
    "git clone https://YOUR_TOKEN@github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)


Cloning into '/content/FIUBench_Reproducing'...



In [2]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

✅ Repository updated
Already up to date.



## Install Dependencies

In [ ]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
!python3.10 -m venv py310_env

import subprocess
from pathlib import Path

VENV_PYTHON = "/content/py310_env/bin/python"
FIUBENCH_DIR = Path('/content/FIUBench_Reproducing/FIUBench')

# Install without flash-attn
subprocess.run(f"{VENV_PYTHON} -m pip install --upgrade pip", shell=True, check=True)

# Read requirements, skip flash-attn
with open(f"{FIUBENCH_DIR}/requirements.txt") as f:
    reqs = [r.strip() for r in f if r.strip() and 'flash-attn' not in r]

with open('/tmp/req.txt', 'w') as f:
    f.write('\n'.join(reqs))

result = subprocess.run(f"{VENV_PYTHON} -m pip install -r /tmp/req.txt", shell=True)
print("✅ Done" if result.returncode == 0 else "❌ Failed")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 https://cli.github.com/packages stable/main amd64 Packages 

## Download SFHQ Images

In [4]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

PROJECT_ROOT = '/content/FIUBench_Reproducing'

try:
    sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
    sfhq_dir.mkdir(parents=True, exist_ok=True)

    existing = list(sfhq_dir.glob("*.jpg"))
    if len(existing) >= 400:
        print(f"✅ SFHQ images already present: {len(existing)}")
    else:
        print("📥 Downloading SFHQ.zip from Hugging Face...")
        zip_path = hf_hub_download(
            repo_id="gray311/FIUBench",
            filename="SFHQ.zip",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )

        extract_dir = sfhq_dir.parent / "sfhq_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"📦 Extracting SFHQ.zip...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        found = list(extract_dir.rglob("*.jpg"))
        print(f"Found {len(found)} jpg files")

        copied = 0
        for src in found:
            dst = sfhq_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1

        total = len(list(sfhq_dir.glob("*.jpg")))
        print(f"✅ SFHQ ready: {total} images")

except Exception as e:
    print(f"❌ SFHQ download failed: {e}")
    raise

📥 Downloading SFHQ.zip from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

📦 Extracting SFHQ.zip...
Found 1000 jpg files
✅ SFHQ ready: 1000 images


In [5]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

# Verify source exists
if not sfhq_src.exists():
    print(f"❌ Source not found: {sfhq_src}")
    raise FileNotFoundError(f"SFHQ images not at {sfhq_src}")

# Clean up old symlink/directory
if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

# Create symlink
sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
if n < 400:
    print(f"⚠️  Only {n} images (expected 400+)")
else:
    print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [6]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## Patches + copy Stage 1 from Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, re, glob
from pathlib import Path

VENV_PYTHON = "/content/py310_env/bin/python"
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'

STAGE1_LOCAL = '/content/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'

os.environ['WANDB_MODE']     = 'online'
os.environ['HYDRA_FULL_ERROR'] = '1'

# ── 1. Copy Stage 1 from Drive ────────────────────────────────────────────────
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print("Copying Stage 1 from Drive...")
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f'rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
    if ret != 0:
        print("Drive sync failed using local path")
safetensors = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert safetensors, f"No safetensors in {STAGE1_LOCAL}"
print(f"✅ Stage 1 ready: {[f.name for f in safetensors]}")

# ── 2. Patch forget.py ────────────────────────────────────────────────────────
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
src = fg_py.read_text()
patched = src

# Remove deepspeed imports (incompatible with torch 2.2.2) - use line-by-line replacement
lines = patched.split('\n')
new_lines = []
skip_count = 0
for i, line in enumerate(lines):
    if skip_count > 0:
        skip_count -= 1
        new_lines.append(line.replace('from transformers.integrations.deepspeed', '# from transformers.integrations.deepspeed').replace('deepspeed_init,', '# deepspeed_init,').replace('deepspeed_load_checkpoint,', '# deepspeed_load_checkpoint,').replace('is_deepspeed_available', '# is_deepspeed_available').replace(')', '# )'))
    elif line.strip() == 'import deepspeed':
        new_lines.append('# ' + line)
        skip_count = 5  # Skip next 5 lines (the from transformers.integrations.deepspeed import block)
    elif skip_count == 0:
        new_lines.append(line)

patched = '\n'.join(new_lines)

# Standard patches
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

# Replace e_prepare_deepspeed function with simplified version
patched = re.sub(
    r'def e_prepare_deepspeed\(model, accelerator\):.*?param\.requires_grad = False',
    '''def e_prepare_deepspeed(model, accelerator):
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    return model''',
    patched,
    flags=re.DOTALL
)

if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py (deepspeed imports removed)")
else:
    print("⚠️  No changes made to forget.py")

# Verify patches
fg_content = fg_py.read_text()
assert '# import deepspeed' in fg_content or 'import deepspeed' not in fg_content, "deepspeed import not removed"
assert 'torch_dtype=torch.bfloat16' in fg_content, "bfloat16 patch missing"
assert 'mixed_precision="bf16"' in fg_content, "mixed_precision patch missing"
print("✅ All forget.py patches verified")

# ── 3. Patch modeling_llava.py (in venv) ──────────────────────────────────────
llava_candidates = glob.glob(
    '/content/py310_env/lib/python*/site-packages/transformers/models/llava/modeling_llava.py'
)
if llava_candidates:
    llava_path = llava_candidates[0]
    llava_src  = Path(llava_path).read_text()
    llava_patched = re.sub(
        r'n_image_tokens != n_image_features',
        'False',
        llava_src
    )
    if llava_patched != llava_src:
        Path(llava_path).write_text(llava_patched)
        print(f"✅ Patched modeling_llava.py (venv)")
    else:
        print("✅ modeling_llava.py already patched")
else:
    print("⚠️  modeling_llava.py not found in venv")

print("\n✅ All patches applied. Ready for Stage 2.")

## Stage 2: Gradient Ascent (GA)

In [20]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Fixing peft/accelerate compatibility...\n")

subprocess.run(
    f"{VENV_PYTHON} -m pip install peft==0.11.1 --force-reinstall -q",
    shell=True, check=True
)

print("✅ peft downgraded to 0.11.1 (compatible with accelerate 0.27.0)")

# Verify
result = subprocess.run(
    f"{VENV_PYTHON} -m pip show peft",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

Fixing peft/accelerate compatibility...

✅ peft downgraded to 0.11.1 (compatible with accelerate 0.27.0)
Name: peft
Version: 0.11.1
Summary: Parameter-Efficient Fine-Tuning (PEFT)
Home-page: https://github.com/huggingface/peft
Author: The HuggingFace team
Author-email: sourab@huggingface.co
License: Apache
Location: /content/py310_env/lib/python3.10/site-packages
Requires: accelerate, huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch, tqdm, transformers
Required-by: 



In [21]:
import subprocess

VENV_PYTHON = "/content/py310_env/bin/python"

print("Downgrading NumPy to 1.23.5...\n")

subprocess.run(
    f"{VENV_PYTHON} -m pip install numpy==1.23.5 --force-reinstall -q",
    shell=True, check=True
)

print("✅ NumPy downgraded")

Downgrading NumPy to 1.23.5...

✅ NumPy downgraded


In [25]:
import subprocess
from pathlib import Path

VENV_PYTHON = "/content/py310_env/bin/python"

print("Fixing all dependency conflicts...\n")

# 1. Uninstall transformers (installed from git, too new)
print("1. Removing transformers from git...")
subprocess.run(f"{VENV_PYTHON} -m pip uninstall -y transformers", shell=True)

# 2. Install compatible transformers version
print("2. Installing transformers 4.41.0 (compatible with torch 2.2.2)...")
subprocess.run(
    f"{VENV_PYTHON} -m pip install transformers==4.41.0 -q",
    shell=True, check=True
)

# 3. Fix peft compatibility by removing problematic import
print("3. Patching peft to remove incompatible accelerate.utils.memory import...")
loftq_file = Path("/content/py310_env/lib/python3.10/site-packages/peft/utils/loftq_utils.py")
if loftq_file.exists():
    content = loftq_file.read_text()
    if "from accelerate.utils.memory import clear_device_cache" in content:
        content = content.replace(
            "from accelerate.utils.memory import clear_device_cache",
            "# from accelerate.utils.memory import clear_device_cache  # Removed for torch 2.2.2 compat"
        )
        # Also comment out the usage if it exists
        content = content.replace(
            "clear_device_cache()",
            "# clear_device_cache()  # Removed for torch 2.2.2 compat"
        )
        loftq_file.write_text(content)
        print("   ✅ Patched peft/utils/loftq_utils.py")
    else:
        print("   ✅ peft already patched")
else:
    print("   ⚠️  peft/utils/loftq_utils.py not found")

# 4. Verify all core packages
print("4. Verifying core packages...")
result = subprocess.run(
    f"{VENV_PYTHON} -c 'import torch, transformers, accelerate, peft; print(f\"torch={{torch.__version__}}, transformers={{transformers.__version__}}, accelerate={{accelerate.__version__}}, peft={{peft.__version__}}\")'",
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print(result.stdout)
    print("✅ All dependencies OK\n")
else:
    print("Import test output:")
    print(result.stderr)
    print(result.stdout)

Fixing all dependency conflicts...

1. Removing transformers from git...
2. Installing transformers 4.41.0 (compatible with torch 2.2.2)...
3. Patching peft to remove incompatible accelerate.utils.memory import...
   ✅ peft already patched
4. Verifying core packages...
torch=2.11.0+cu130, transformers=4.41.0, accelerate=1.13.0, peft=0.11.1

✅ All dependencies OK



In [26]:
from pathlib import Path

eval_py = Path('/content/FIUBench_Reproducing/FIUBench/evaluate_util.py')
content = eval_py.read_text()
content = content.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
eval_py.write_text(content)
print("✅ Patched to use sdpa")

✅ Patched to use sdpa


In [27]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'ga'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"bash -c 'source /content/py310_env/bin/activate && "
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29510 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true' "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ GA checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ GA training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ GA failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


Starting Stage 2 [GA]  lr=2e-5  batch=8  accum=16  epochs=8  split=forget5
/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/content/py310_env/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[2026-04-19 17:35:02,995] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package w

## Stage 2: Gradient Difference (GD)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'gd'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29511 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ GD checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ GD training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ GD failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


## Stage 2: KL Minimization (KL)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'kl'
LR           = '1e-4'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29512 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{METHOD}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ KL checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ KL training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ KL failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


## Stage 2: Preference Optimization / IDK (PO)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'idk'
LABEL        = 'po'
LR           = '3e-4'
STAGE2_LOCAL = f'/content/stage2_{LABEL}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{LABEL}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29513 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{LABEL}_log.txt"
)

print(f"Starting Stage 2 [PO/IDK]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    os.system(f"cp /tmp/forget_{LABEL}_log.txt {STAGE2_DRIVE}/training_log.txt")
    print(f"✅ PO checkpoints saved to {STAGE2_DRIVE}")
    print(f"✅ PO training log saved to {STAGE2_DRIVE}/training_log.txt")
else:
    print(f"❌ PO failed (exit {ret}). Check /tmp/forget_{LABEL}_log.txt")


## Retain model baseline

In [ ]:
# The retain model is fine-tuned on S_R only (never sees forget5 identities).
# It is the ideal unlearning upper bound used for KS-Test in Day 5.
# Training: same as Stage 1 but data filtered to retain95 identities only.

import os, subprocess, time, json
from pathlib import Path

PROJECT_ROOT   = '/content/FIUBench_Reproducing'
FIUBENCH_DIR   = f'{PROJECT_ROOT}/FIUBench'
PRETRAINED_ID  = 'xtuner/llava-phi-3-mini-hf'   # start from pretrained, not Stage 1
RETAIN_LOCAL   = '/content/retain_model'
RETAIN_DRIVE   = '/content/drive/MyDrive/fiubench_checkpoints/retain_model'
DATA_PATH      = f'{FIUBENCH_DIR}/dataset/full.json'
RETAIN_DATA    = '/tmp/retain95.json'

Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)

# Build retain95 dataset (identities NOT in forget5)
with open(DATA_PATH) as f:
    full_data = [json.loads(line) for line in f if line.strip()]

splits_path = Path(PROJECT_ROOT) / 'FIUBench/dataset/split.json'  # file is split.json, not splits.json
if splits_path.exists():
    with open(splits_path) as f:
        splits = json.load(f)
    forget5_ids = set(splits['forget5'])
else:
    all_ids = list({x.get('unique_id', x.get('image_path','')) for x in full_data})
    forget5_ids = set(all_ids[:20])

retain95_data = [x for x in full_data
                 if x.get('unique_id', x.get('image_path','')) not in forget5_ids]
print(f"Retain95 identities: {len({x.get('unique_id','') for x in retain95_data})}")
print(f"Retain95 QA pairs:   {len(retain95_data)}")

with open(RETAIN_DATA, 'w') as f:
    json.dump(retain95_data, f)
print(f"✅ Saved retain95 dataset to {RETAIN_DATA}")

# Fine-tune on retain95 using same Stage 1 hyperparameters
# lr=2e-5, batch=8, accum=16, epochs=10, full fine-tuning (no LoRA)
cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29514 finetune.py "
    f"--config-name finetune_llava_phi "
    f"save_dir={RETAIN_LOCAL} "
    f"data_path={RETAIN_DATA} "
    f"split=full "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/retain_model_log.txt"
)
# Note: model_id, lr=2e-5, num_epochs=10, seed=0 are correct yaml defaults — no override needed

print("\nStarting retain model fine-tuning on S_R only...")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(RETAIN_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {RETAIN_LOCAL}/ {RETAIN_DRIVE}/")
    print(f"✅ Retain model saved to {RETAIN_DRIVE}")
else:
    print(f"❌ Retain model training failed. Check /tmp/retain_model_log.txt")


## Stage 2 Evaluation - FIUBench Metrics (forget5)

Runs evaluate_util.py on each checkpoint (retain + GA/GD/KL/PO), then aggregates with aggregate_eval_stat.py.

Output per method: Forget Quality (KS-Test p-value), Model Utility (ROUGE/Prob/Truth Ratio harmonic mean).

In [21]:
import os, subprocess, shutil
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_DRIVE = f'{DRIVE_ROOT}/retain_model'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Patch evaluate_util.py ----------------------------------------------------
_eu = Path(f"{FIUBENCH_DIR}/evaluate_util.py")
_eu_src = _eu.read_text()
_eu_new = _eu_src
_eu_new = _eu_new.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
_eu_new = _eu_new.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
_eu_new = _eu_new.replace('model.half().cuda()', 'model.cuda()')
if _eu_new != _eu_src:
    _eu.write_text(_eu_new)
    print('Patched evaluate_util.py: flash_attention_2->sdpa, float16->bfloat16, .half() removed')
else:
    print('evaluate_util.py already patched')

# -- Restore Stage 1 from Drive -------------------------------------------------
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print('Restoring stage1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {DRIVE_ROOT}/stage1/ {STAGE1_LOCAL}/")
else:
    print(f'Stage1 at {STAGE1_LOCAL}')

# -- Restore retain model from Drive --------------------------------------------
if not Path(RETAIN_LOCAL).exists() or not list(Path(RETAIN_LOCAL).glob('*.safetensors')):
    print('Restoring retain model from Drive...')
    Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {RETAIN_DRIVE}/ {RETAIN_LOCAL}/")
else:
    print(f'Retain model at {RETAIN_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -------------------------------------
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        print(f"Restoring {method} from Drive...")
        Path(paths['local']).mkdir(parents=True, exist_ok=True)
        os.system(f"rsync -ah {paths['drive']}/ {paths['local']}/")
    else:
        print(f"{method} checkpoint at {ckpt}")

evaluate_util.py already patched
Stage1 at /content/stage1_final
Retain model at /content/retain_model
ga checkpoint at /content/stage2_ga/checkpoint.pt
gd checkpoint at /content/stage2_gd/checkpoint.pt
kl checkpoint at /content/stage2_kl/checkpoint.pt
po checkpoint at /content/stage2_po/checkpoint.pt


In [ ]:
# Evaluate retain model on forget5 + retain5 splits.
# This produces the reference distribution for KS-Test (Forget Quality).
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_EVAL  = f'{RETAIN_LOCAL}/eval_results'
Path(RETAIN_EVAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"bash -c 'set -o pipefail; cd {FIUBENCH_DIR} && "
    f"python evaluate_util.py --config-name eval "
    f"model_path={RETAIN_LOCAL} "
    f"save_dir={RETAIN_EVAL} "
    f"'LoRA.r=0' "
    f"'hydra.run.dir=/tmp/hydra_eval_retain' "
    f"2>&1 | tee /tmp/eval_retain_log.txt'"
)

print('Evaluating retain model (forget5 + retain5)...')
print('=' * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {proc.returncode}  |  Time: {h}h {m}m {s}s")

agg = Path(RETAIN_EVAL) / 'retain5_eval_log_aggregated.json'
print(f"{'PASS' if agg.exists() else 'FAIL'} Retain eval log: {agg}")

# --- forget5 full eval (needed for KS-Test) ---
print(f"\nRunning forget5 full-metric eval for {method.upper()}...")
t1 = time.time()
proc2 = subprocess.Popen(
    ['python', 'evaluate_util.py', '--config-name', 'eval',
        f'model_path={STAGE1_LOCAL}',
        'LoRA.r=128', 'LoRA.alpha=256',
        f'LoRA.lora_path={ckpt}',
        f'save_dir={method_dir}/eval_results/',
        'split_list=[forget5]', 'eval_task=[eval_retain_log]',
        'robust_eval=[[rouge]]',
        'batch_size=4', 'perturb_batch_size=4', 'overwrite=true',
        f'hydra.run.dir=/tmp/hydra_{method}_forget5_full'],
    cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc2.stdout:
    print(line, end='', flush=True)
proc2.wait()
elapsed2 = time.time() - t1
h, m, s = int(elapsed2//3600), int((elapsed2%3600)//60), int(elapsed2%60)
print(f"Exit: {proc2.returncode}  |  Time: {h}h {m}m {s}s")
forget_full = Path(method_dir) / 'eval_results' / 'forget5_eval_retain_log.json'
print(f"{'PASS' if forget_full.exists() else 'FAIL'} {method.upper()} forget5 full log: {forget_full}")

Evaluating retain model (forget5 + retain5)...
2026-04-15 16:22:32.148535: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 16:22:32.177338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776270152.200961   23564 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776270152.208449   23564 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776270152.232840   23564 computation_placer.cc:177] computation placer already regi

In [22]:
# Evaluate each unlearning method on forget5 + retain5.
# Restores Stage 2 checkpoints from Drive at the start — safe to re-run after runtime restart.
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Restore Stage 1 if missing -----------------------------------------------
if not list(Path(STAGE1_LOCAL).glob('*.safetensors') if Path(STAGE1_LOCAL).exists() else []):
    print('Restoring Stage 1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f'rsync -ah {DRIVE_ROOT}/stage1/ {STAGE1_LOCAL}/')
    assert ret == 0, 'rsync stage1 failed'
print(f'Stage 1 ready: {STAGE1_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -----------------------------------
available = []
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        drive_src = paths['drive']
        if Path(drive_src).exists():
            print(f'Restoring {method} from Drive...')
            Path(paths['local']).mkdir(parents=True, exist_ok=True)
            ret = os.system(f"rsync -ah {drive_src}/ {paths['local']}/")
            if ret == 0 and ckpt.exists():
                print(f'  ✅ {method}: checkpoint.pt restored')
                available.append(method)
            else:
                print(f'  ❌ {method}: rsync returned {ret} or checkpoint.pt missing after sync')
        else:
            print(f'  ⚠️  {method}: not on Drive at {drive_src} — skipping')
    else:
        print(f'  ✅ {method}: checkpoint.pt already present')
        available.append(method)

print(f'
Methods to evaluate: {available}')

# -- Run evaluation for each available method ---------------------------------
for method in available:
    method_dir = METHODS[method]['local']
    ckpt       = f"{method_dir}/checkpoint.pt"
    save_dir   = f"{method_dir}/eval_results/"
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # --- retain5 + forget5 standard eval (EM, ROUGE, GPT, Truth Ratio) -------
    print(f"
{'='*70}")
    print(f"Evaluating {method.upper()} — retain5 + forget5 standard eval...")
    print(f"{'='*70}")
    t0 = time.time()
    proc = subprocess.Popen(
        ['python', 'evaluate_util.py', '--config-name', 'eval',
         f'model_path={STAGE1_LOCAL}',
         'LoRA.r=128', 'LoRA.alpha=256', f'LoRA.lora_path={ckpt}',
         f'save_dir={save_dir}',
         f'hydra.run.dir=/tmp/hydra_eval_{method}'],
        cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    print(f"
Exit: {proc.returncode}  |  Time: {h}h {m}m {s}s")
    agg = Path(save_dir) / 'retain5_eval_log_aggregated.json'
    print(f"{'PASS' if agg.exists() else 'FAIL'} Retain5 eval log: {agg}")

    # --- forget5 full-metric eval (MIA, APE, KS-Test inputs) -----------------
    print(f"
Running forget5 full-metric eval for {method.upper()}...")
    t1 = time.time()
    proc2 = subprocess.Popen(
        ['python', 'evaluate_util.py', '--config-name', 'eval',
         f'model_path={STAGE1_LOCAL}',
         'LoRA.r=128', 'LoRA.alpha=256', f'LoRA.lora_path={ckpt}',
         f'save_dir={save_dir}',
         'split_list=[forget5]', 'eval_task=[eval_retain_log]',
         'robust_eval=[[rouge]]',
         'batch_size=4', 'perturb_batch_size=4', 'overwrite=true',
         f'hydra.run.dir=/tmp/hydra_{method}_forget5_full'],
        cwd=FIUBENCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc2.stdout:
        print(line, end='', flush=True)
    proc2.wait()
    elapsed2 = time.time() - t1
    h, m, s = int(elapsed2//3600), int((elapsed2%3600)//60), int(elapsed2%60)
    print(f"Exit: {proc2.returncode}  |  Time: {h}h {m}m {s}s")
    forget_full = Path(save_dir) / 'forget5_eval_retain_log.json'
    print(f"{'PASS' if forget_full.exists() else 'FAIL'} Forget5 full log: {forget_full}")


Stage 1 ready: /content/stage1_final
  ✅ ga: checkpoint.pt already present
  ✅ gd: checkpoint.pt already present
  ✅ kl: checkpoint.pt already present
  ✅ po: checkpoint.pt already present

Methods to evaluate: ['ga', 'gd', 'kl', 'po']

Evaluating GA (forget5 + retain5)...
2026-04-15 17:25:13.978594: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 17:25:14.006060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776273914.028693   39419 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776273914.036453   39419 cuda_blas

## Save results to drive

In [26]:
import subprocess, os

DRIVE_STAGE2 = "/content/drive/MyDrive/fiubench_checkpoints/stage2"
LOCAL_STAGE2 = "/content"

for method in ["ga", "gd", "kl", "po"]:
    src = f"{LOCAL_STAGE2}/stage2_{method}/eval_results/"
    dst = f"{DRIVE_STAGE2}/stage2_{method}/eval_results/"
    os.makedirs(dst, exist_ok=True)
    ret = subprocess.call(["rsync", "-av", src, dst])
    print(f"{method}: rsync returned {ret}")


ga: rsync returned 0
gd: rsync returned 0
kl: rsync returned 0
po: rsync returned 0


## Calculate Results

In [ ]:
# Aggregate results: compute Forget Quality (KS-Test) + Model Utility for each method.
import os, json, csv, subprocess
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
RESULTS_DIR  = '/content/eval_results'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

RETAIN_DIR = '/content/retain_model/eval_results'
METHODS = {
    'GA': '/content/stage2_ga/eval_results',
    'GD': '/content/stage2_gd/eval_results',
    'KL': '/content/stage2_kl/eval_results',
    'PO': '/content/stage2_po/eval_results',
}

def build_combined_json(eval_dir, tmp_name):
    """Combine forget5_eval_retain_log + retain5_eval_retain_log into the
    format aggregate_eval_stat.py expects."""
    forget_path = Path(eval_dir) / 'forget5_eval_retain_log.json'
    retain_path = Path(eval_dir) / 'retain5_eval_retain_log.json'
    if not forget_path.exists():
        print(f"  MISSING: {forget_path}")
        return None
    if not retain_path.exists():
        print(f"  MISSING: {retain_path}")
        return None
    combined = {
        'eval_forget_log.json': json.load(open(forget_path)),
        'eval_retain_log.json': json.load(open(retain_path)),
    }
    tmp_path = f'/tmp/{tmp_name}_combined.json'
    with open(tmp_path, 'w') as f:
        json.dump(combined, f)
    return tmp_path

# Build retain reference combined JSON
retain_combined = build_combined_json(RETAIN_DIR, 'retain')
if not retain_combined:
    print("Retain eval incomplete — run cells 23 first.")
else:
    all_results = []
    for method, eval_dir in METHODS.items():
        ckpt_combined = build_combined_json(eval_dir, method.lower())
        if not ckpt_combined:
            print(f"Skipping {method}: forget5 full eval not yet run.")
            continue

        save_csv = f"{RESULTS_DIR}/{method.lower()}_aggr.csv"
        result = subprocess.run(
            ['python', 'aggregate_eval_stat.py',
             f'retain_result={retain_combined}',
             f'ckpt_result={ckpt_combined}',
             f'method_name={method}',
             f'save_file={save_csv}',
             f'hydra.run.dir=/tmp/hydra_agg_{method.lower()}'],
            cwd=FIUBENCH_DIR, capture_output=True, text=True
        )
        if result.returncode == 0 and Path(save_csv).exists():
            rows = list(csv.DictReader(open(save_csv)))
            if rows:
                all_results.append(rows[0])
                print(f"✅ {method}: aggregated")
        else:
            print(f"❌ {method}: aggregate_eval_stat.py failed")
            print(result.stderr[-500:])

    if all_results:
        print('\n' + '='*90)
        print('FIUBench Reproduced Results (forget5)')
        print('='*90)
        keys = ['Method', 'Forget Quality', 'Model Utility',
                'ROUGE Forget', 'ROUGE Retain', 'Prob. Forget', 'Prob. Retain',
                'Truth Ratio Forget', 'Truth Ratio Retain', 'GPT Retain', 'EM Retain']
        header = f"{'Method':<8}" + ''.join(f"  {k:<20}" for k in keys[1:])
        print(header)
        print('-' * len(header))
        for r in all_results:
            row = f"{r.get('Method',''):<8}"
            for k in keys[1:]:
                v = r.get(k, 'N/A')
                try:    row += f"  {float(v):<20.4f}"
                except: row += f"  {str(v):<20}"
            print(row)
        print('='*90)

        DRIVE_RESULTS = '/content/drive/MyDrive/fiubench_checkpoints/eval_results_forget5'
        os.system(f"rsync -ah {RESULTS_DIR}/ {DRIVE_RESULTS}/")
        print(f"\nResults synced to Drive: {DRIVE_RESULTS}")
    else:
        print("\nNo results — run forget5 full eval in cells 23 and 24 first.")

  MISSING: /content/retain_model/eval_results/forget5_eval_retain_log.json
Retain eval incomplete — run cells 23 first.
